# 15 OpenOximetry XAI Explanation Poster Plot

Generate a publication-ready local feature contribution plot for the poster architecture infographic placeholder `n19`.

The figure shows per-feature contributions for a single low-oxygen-risk window, computed as `standardized_feature_value * coefficient`. Positive contributions (orange) push toward the low-oxygen-risk class; negative contributions (blue) push toward the normal class.

Input: `outputs/openox_canonical_window_features.csv` (notebook 10), or fallback demo values from the paper.
Outputs:
- `outputs/poster/xai-explanation-plot.svg`
- `outputs/poster/xai-explanation-plot.png`
- `outputs/poster/xai-explanation-plot-poster.svg` (compact poster variant)
- `outputs/poster/xai-explanation-plot-poster.png`

In [ ]:
from __future__ import annotations

import csv
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

SEED = 7
PROJECT_ROOT = (Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve())
OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'poster'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FEATURE_NAMES = [
    'mean', 'std', 'minimum', 'maximum', 'value_range', 'rms',
    'skewness', 'kurtosis', 'energy', 'derivative_std',
    'dominant_frequency_hz', 'bandpower_0_5_2_hz', 'bandpower_2_5_hz',
    'bandpower_ratio_0_5_2_to_2_5',
]

N_FEATURES = len(FEATURE_NAMES)
print(f"Project root: {PROJECT_ROOT}")
print(f"Output dir: {OUTPUT_DIR}")
print(f"Feature count: {N_FEATURES}")

### Data Loading

Try to load the canonical OpenOximetry window features CSV. If unavailable, use a fallback demo dataset constructed from the class-balanced distribution reported in the research plan.

In [ ]:
feature_csv = PROJECT_ROOT / 'outputs' / 'openox_canonical_window_features.csv'

if feature_csv.exists():
    print(f"Loading canonical features from {feature_csv}")
    df = pd.read_csv(feature_csv)
    for col in FEATURE_NAMES:
        df[col] = pd.to_numeric(df[col], errors='raise')
    df['recording_id'] = df['recording_id'].astype(str)
    df['label'] = pd.to_numeric(df['label'], errors='raise').astype(int)
    USE_DEMO = False
else:
    print(f"Canonical CSV not found at {feature_csv}")
    print("Falling back to demo values from the paper.")
    USE_DEMO = True

print(f"USE_DEMO = {USE_DEMO}")

### Train Logistic Regression or Load Demo Coefficients

If real data is available, train a single grouped-fold Logistic Regression and extract coefficients. Otherwise use the mean coefficients reported in the paper.

In [ ]:
if not USE_DEMO:
    groups = df['recording_id'].values
    X = df[FEATURE_NAMES].values.astype(np.float64)
    y = df['label'].values

    unique_groups = np.unique(groups)
    np.random.seed(SEED)
    perm = np.random.permutation(len(unique_groups))
    n_train = int(0.7 * len(unique_groups))
    train_groups = set(unique_groups[perm[:n_train]])

    train_idx = np.array([i for i, g in enumerate(groups) if g in train_groups])
    test_idx = np.array([i for i, g in enumerate(groups) if g not in train_groups])

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X[train_idx])
    X_test = scaler.transform(X[test_idx])
    y_train = y[train_idx]

    model = LogisticRegression(
        class_weight='balanced', max_iter=1000, solver='liblinear', random_state=SEED
    )
    model.fit(X_train, y_train)

    coef = model.coef_[0]
    intercept = model.intercept_[0]
    y_test = y[test_idx]
    probs = model.predict_proba(X_test)[:, 1]

    positive_test_idx = np.where(y_test == 1)[0]
    if len(positive_test_idx) > 0:
        example_pos = positive_test_idx[0]
        example_features = X_test[example_pos]
        example_prob = probs[example_pos]
    else:
        example_pos = 0
        example_features = X_test[example_pos]
        example_prob = probs[example_pos]

    contributions = example_features * coef
    print(f"Trained on {len(train_idx)} rows, evaluated on {len(test_idx)} rows")
    print(f"Example window risk probability: {example_prob:.4f}")
else:
    coef = np.array([
        -0.055, -0.149,  0.057,  0.032,  0.028, -0.041,
        -0.023,  0.039, -0.018,  0.051,  0.015, -0.012,
         0.008, -0.006
    ])
    intercept = 0.096
    example_features = np.array([
         0.5,  1.0,  1.0,  0.5,  0.3,  0.5,
         0.3,  0.5,  0.3,  1.0,  0.5,  0.5,
         0.5,  0.5
    ])
    contributions = example_features * coef
    log_odds = contributions.sum() + intercept
    example_prob = 1.0 / (1.0 + np.exp(-log_odds))
    print(f"Demo example risk probability: {example_prob:.4f}")

print(f"\nCoefficients (top 5 by |value|):")
top5_idx = np.argsort(np.abs(coef))[::-1][:5]
for i in top5_idx:
    print(f"  {FEATURE_NAMES[i]:30s}  {coef[i]:+.4f}")
print(f"\nExample contributions (top 5 by |contribution|):")
top5_contrib = np.argsort(np.abs(contributions))[::-1][:5]
for i in top5_contrib:
    print(f"  {FEATURE_NAMES[i]:30s}  {contributions[i]:+.4f}")

### Render Poster-Quality XAI Explanation SVG

Horizontal bar chart showing per-feature contributions for the example low-oxygen-risk window.
Risk-increasing contributions in orange, risk-reducing in blue.
Output dimensions: 8 x 4 in at 150 DPI.

In [ ]:
plt.style.use('default')

RISK_COLOR = '#E65100'
NORMAL_COLOR = '#1565C0'
NEUTRAL_GREY = '#666666'
LIGHT_GREY = '#CCCCCC'

LABEL_MAP = {
    'mean': 'Mean amplitude',
    'std': 'PPG variability',
    'minimum': 'Min amplitude',
    'maximum': 'Max amplitude',
    'value_range': 'Amplitude range',
    'rms': 'Signal power',
    'skewness': 'Skewness',
    'kurtosis': 'Kurtosis',
    'energy': 'Energy',
    'derivative_std': 'Slope variability',
    'dominant_frequency_hz': 'Dominant freq.',
    'bandpower_0_5_2_hz': 'Bandpower (0.5-2 Hz)',
    'bandpower_2_5_hz': 'Bandpower (2-5 Hz)',
    'bandpower_ratio_0_5_2_to_2_5': 'Bandpower ratio',
}

pred_label = 'Low-oxygen-risk' if example_prob >= 0.5 else 'Normal'

top_n = 8
order = np.argsort(np.abs(contributions))[::-1][:top_n]
plot_features_raw = [FEATURE_NAMES[i] for i in order]
plot_features = [LABEL_MAP.get(f, f) for f in plot_features_raw]
plot_contributions = contributions[order]
bar_colors = [RISK_COLOR if v > 0 else NORMAL_COLOR for v in plot_contributions]

fig, ax = plt.subplots(figsize=(8, 3.6))

bars = ax.barh(range(len(plot_features)), plot_contributions,
               color=bar_colors, height=0.6, edgecolor='white', linewidth=0.5)

ax.axvline(0, color=NEUTRAL_GREY, linewidth=0.8, linestyle='-')

ax.set_yticks(range(len(plot_features)))
ax.set_yticklabels(plot_features, fontsize=10)
ax.set_xlabel('Feature contribution (standardized)', fontsize=10, fontweight='bold')
ax.set_title('Local XAI Explanation', fontsize=12, fontweight='bold', pad=8)

ax.tick_params(axis='x', labelsize=9)
ax.xaxis.set_major_locator(ticker.MaxNLocator(6))

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_color(LIGHT_GREY)
ax.spines['bottom'].set_color(LIGHT_GREY)

ax.margins(y=0.15)
ax.grid(True, axis='x', alpha=0.15, linestyle='-')

ax.text(0.02, 0.98, 'Toward low-oxygen-risk', transform=ax.transAxes,
        fontsize=8, color=RISK_COLOR, va='top', ha='left', fontweight='bold')
ax.text(0.02, 0.93, 'Toward normal', transform=ax.transAxes,
        fontsize=8, color=NORMAL_COLOR, va='top', ha='left', fontweight='bold')

info_text = (
f"    Risk probability: {example_prob:.2f}  |  Predicted: {pred_label}\n"
"    Engineering flag only, not clinical diagnosis"
)
fig.text(0.5, -0.01, info_text, ha='center', va='top', fontsize=9,
         color=NEUTRAL_GREY, fontstyle='italic')

plt.tight_layout(pad=1.0, rect=[0, 0.06, 1, 1])

svg_path = OUTPUT_DIR / 'xai-explanation-plot.svg'
png_path = OUTPUT_DIR / 'xai-explanation-plot.png'
plt.savefig(svg_path, dpi=150, bbox_inches='tight')
plt.savefig(png_path, dpi=200, bbox_inches='tight')
plt.close()

print(f'SVG: {svg_path} ({svg_path.stat().st_size} bytes)')
print(f'PNG: {png_path} ({png_path.stat().st_size} bytes)')

### Generate Poster-Optimized Compact Variant

Compact version for the Draw.io placeholder n19. Shows only top 6 features with minimal axis labels.

In [ ]:
poster_top_n = 5
poster_order = np.argsort(np.abs(contributions))[::-1][:poster_top_n]
poster_features_raw = [FEATURE_NAMES[i] for i in poster_order]
poster_features = [LABEL_MAP.get(f, f) for f in poster_features_raw]
poster_contributions = contributions[poster_order]
poster_colors = [RISK_COLOR if v > 0 else NORMAL_COLOR for v in poster_contributions]

fig2, ax2 = plt.subplots(figsize=(3.2, 1.6))

ax2.barh(range(len(poster_features)), poster_contributions,
         color=poster_colors, height=0.6, edgecolor='white', linewidth=0.3)

ax2.axvline(0, color='#888888', linewidth=0.6, linestyle='-')

ax2.set_yticks(range(len(poster_features)))
ax2.set_yticklabels(poster_features, fontsize=6.5)
ax2.set_xticks([])
ax2.tick_params(axis='y', pad=1)
ax2.set_title('Local XAI Explanation', fontsize=7, color='#444444',
              fontweight='bold', pad=1.5)

ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)
ax2.spines['left'].set_color(LIGHT_GREY)
ax2.spines['bottom'].set_color(LIGHT_GREY)
ax2.tick_params(bottom=False, left=False, colors='#888888')
ax2.grid(True, axis='x', alpha=0.08, linestyle='-')

risk_annotation = f"Risk probability = {example_prob:.2f}"
ax2.text(0.5, -0.35, risk_annotation, transform=ax2.transAxes,
         ha='center', fontsize=6, color='#666666', fontstyle='italic')

plt.tight_layout(pad=0.2)

poster_svg = OUTPUT_DIR / 'xai-explanation-plot-poster.svg'
poster_png = OUTPUT_DIR / 'xai-explanation-plot-poster.png'
plt.savefig(poster_svg, dpi=200, bbox_inches='tight', pad_inches=0.06, transparent=False)
plt.savefig(poster_png, dpi=300, bbox_inches='tight', pad_inches=0.06)
plt.close()

print(f'Poster SVG: {poster_svg} ({poster_svg.stat().st_size} bytes)')
print(f'Poster PNG: {poster_png} ({poster_png.stat().st_size} bytes)')

### Verify Output

The plot shows the top contributing features for an example low-oxygen-risk window.
Orange bars indicate features that push the prediction toward risk (positive contribution);
blue bars indicate features that push toward normal (negative contribution).

This matches the compact explanation string used in the project:
"Low-oxygen-risk flag, risk probability 0.51. Strongest model evidence: std=-0.149, minimum=+0.057, derivative_std=+0.051."

In [ ]:
print("Output files:")
for f in sorted(OUTPUT_DIR.glob('xai-explanation-plot*')):
    print(f"  {f.name:55s} {f.stat().st_size:>8d} bytes")

## Notes

- This figure uses **Logistic Regression coefficient-based explanations** (standardized feature value * coefficient).
- SHAP was not required because Logistic Regression is intrinsically transparent.
- The explanation-clarity rubric (research plan Table 11) requires: compactness, consistency with model behavior, and practical usefulness.
- The figure serves placeholder n19 in the poster architecture infographic.